# Lab 3 (part A) - Auto Loader

Lab 2 loaded three big files. Here the source is a folder with ~1000 small files.

Plan: make the 1000 files, let Auto Loader eat them batch by batch, then look at how it went.
Schema evolution, triggers and reloading come further down.

In [0]:

import json

from pyspark.sql.types import StructType, StructField, LongType
from datetime import datetime, timezone
from pyspark.sql import functions as F

dbutils.widgets.text("login", "")
dbutils.widgets.text("target_catalog", "")
dbutils.widgets.text("storage_account", "")
dbutils.widgets.text("container", "")

login           = dbutils.widgets.get("login")
catalog         = dbutils.widgets.get("target_catalog")
storage_account = dbutils.widgets.get("storage_account")
container       = dbutils.widgets.get("container")

assert all([login, catalog, storage_account, container])

bronze = f"{login}_bronze"
abfss  = f"abfss://{container}@{storage_account}.dfs.core.windows.net"


In [0]:
spark.sql(f"""CREATE EXTERNAL VOLUME IF NOT EXISTS {catalog}.{bronze}.landing_stream
              LOCATION '{abfss}/landing_stream'""")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog}.{bronze}.checkpoints")

landing_stream = f"/Volumes/{catalog}/{bronze}/landing_stream"
chk            = f"/Volumes/{catalog}/{bronze}/checkpoints"
source_dir     = f"{landing_stream}/raw_events"

## 1. Make the 1000 files

Uploading 1000 files from a laptop would take forever, so we make them here: read the bronze
table from Lab 2, throw away the ingestion metadata to get the original records back, and split
it into 1000 partitions - one file each. Real files, real folder in ADLS.

Every run drops its files into its own `batch_ts=...` folder. Two reasons. Spark names parts
with a random UUID, so without the stamp you can't tell later which generation a record came
from. And Auto Loader keys on the **full path** - a name that happened to repeat would be
silently skipped as "already seen". Stamped folders make the path unique and come out as a
Hive-style partition column for free.


In [0]:
N_FILES = 1000


def generate_batch(df, n_files=N_FILES, stamp=None, base=None):
    """Write df as n_files json files into their own batch_ts=... folder. Returns the folder."""
    base  = base or source_dir
    stamp = stamp or datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    out   = f"{base}/batch_ts={stamp}"
    df.repartition(n_files).write.mode("overwrite").format("json").save(out)

    files = [f for f in dbutils.fs.ls(out) if f.name.endswith(".json")]
    print(f"{out} -> {len(files)} files, {sum(f.size for f in files) / 1024**2:.1f} MB")
    return out

src = (spark.table(f"{catalog}.{bronze}.raw_events_bronze")
       .drop("source_file", "ingestion_ts", "load_date", "_rescued_data"))

generate_batch(src)

## 2. Let Auto Loader read them

Four options carry this whole lab:

- `schemaLocation` - where the inferred schema is kept. No schema on disk, nothing to evolve later.
- `schemaEvolutionMode` - what happens when a new column shows up. Default `addNewColumns` stops
  the stream, writes down the wider schema, and the next run just works.
- `rescuedDataColumn` - anything that doesn't fit the schema lands here instead of vanishing.
- `maxFilesPerTrigger` - files per micro-batch. This is what splits our 1000 files into ~10 batches
  instead of one giant one.

`mergeSchema` on the writer side is so Delta accepts the wider schema when it changes.
`availableNow` means: process whatever is there, in several batches, then stop - cheap for a job.

In [0]:
target     = f"{catalog}.{bronze}.raw_events_stream_bronze"
ckpt       = f"{chk}/raw_events_stream"
schema_loc = f"{ckpt}/schema"


def run_autoloader(max_files_per_trigger=100, evolution_mode="addNewColumns"):
    """One availableNow pass over source_dir. Returns the query so we can read its stats after."""
    q = (spark.readStream.format("cloudFiles")
         .option("cloudFiles.format", "json")
         .option("cloudFiles.schemaLocation", schema_loc)
         .option("cloudFiles.schemaEvolutionMode", evolution_mode)
         .option("cloudFiles.maxFilesPerTrigger", max_files_per_trigger)
         .option("rescuedDataColumn", "_rescued_data")
         .load(source_dir)
         .withColumn("source_file",   F.col("_metadata.file_path"))
         .withColumn("source_file_ts", F.col("_metadata.file_modification_time"))
         .withColumn("ingestion_ts",  F.current_timestamp())
         .withColumn("load_date",     F.current_date())
         .writeStream
         .option("checkpointLocation", ckpt)
         .option("mergeSchema", "true")
         .trigger(availableNow=True)
         .toTable(target))
    q.awaitTermination()
    return q


query = run_autoloader()
print(target, "->", spark.table(target).count(), "rows")

In [0]:
display(spark.table(target).limit(5))

In [0]:
display(spark.table(target).groupBy("batch_ts").count())

## 3. How did it go - files per batch

`recentProgress` has one entry per micro-batch. `numFilesOutstanding` is the backlog *left over*,
so the drop between two batches tells us how many files that batch actually took. Should sit
right around `maxFilesPerTrigger`.

In [0]:
STATS_SCHEMA = StructType([
    StructField("batch_id",        LongType()),
    StructField("rows",            LongType()),
    StructField("files_consumed",  LongType()),
    StructField("files_remaining", LongType()),
    StructField("duration_ms",     LongType()),
])

def _int(v):
    return None if v is None else int(v)

def progress_stats(q):
    rows, prev_backlog = [], None
    for p in q.recentProgress:
        d = p if isinstance(p, dict) else json.loads(p.json)
        backlog = int(d["sources"][0].get("metrics", {}).get("numFilesOutstanding", 0))
        duration = d.get("durationMs", {}).get("triggerExecution")
        rows.append((
            _int(d.get("batchId")),
            _int(d["numInputRows"] if d.get("numInputRows") is not None else d.get("sources", [{}])[0].get("numInputRows")),
            None if prev_backlog is None else prev_backlog - backlog,
            backlog,
            _int(duration),
        ))

        prev_backlog = backlog
    return rows


stats = progress_stats(query)
print(f"micro-batches: {len(stats)}")
display(spark.createDataFrame(stats, STATS_SCHEMA))

In [0]:
display(spark.sql(f"DESCRIBE HISTORY {target}")
        .select("version", "timestamp", "operation",
                F.col("operationMetrics.numOutputRows").alias("rows"),
                F.col("operationMetrics.numAddedFiles").alias("output_files"))
        .orderBy("version"))

### Small files, the other end

Every micro-batch is its own commit, and every commit writes its own files - so a stream that runs
often leaves behind a pile of tiny ones, and every later query has to open all of them.

`maxFilesPerTrigger` handles the **input** side. This is the **output** side, and it's Delta's job:

- `optimizeWrite` reshuffles before writing so the files come out a sensible size
- `autoCompact` checks after each commit and merges the small ones in the background
- `OPTIMIZE` does the same to what's already there, on demand

In [0]:
def file_count(t):
    return spark.sql(f"DESCRIBE DETAIL {t}").select("numFiles").first()[0]

print("before:", file_count(target))

spark.sql(f"""ALTER TABLE {target} SET TBLPROPERTIES (
    'delta.autoOptimize.optimizeWrite' = 'true',
    'delta.autoOptimize.autoCompact'   = 'true')""")
spark.sql(f"OPTIMIZE {target}")

print("after :", file_count(target))


## 4. Schema evolution

The source system adds a field one day - here a `priority` column - and we
want the pipeline to survive it without anyone editing the notebook.

We're on `addNewColumns`, the default. What happens is deliberately dramatic: the stream **fails**
with `UnknownFieldException`, but on the way out it writes the wider schema into `schemaLocation`.
The next run starts from the new schema and just works. In a job that means one failed run and a
green retry - which is why streaming jobs are normally configured to retry.

In [0]:
generate_batch(src.limit(50_000).withColumn("priority", F.lit("high")), n_files=20)

In [0]:
try:
    run_autoloader()
    print("finished without failing - the schema already knew this column")
except Exception as e:
    msg = str(e)
    i = msg.find("UnknownFieldException")
    print(f"{type(e).__name__} - stream stopped as expected\n")
    print(msg[i:i + 500] if i > -1 else msg[:500])

In [0]:
display(dbutils.fs.ls(f"{schema_loc}/_schemas"))

In [0]:
query_evolved = run_autoloader()
print(target, "->", spark.table(target).count(), "rows")

display(spark.table(target).groupBy("batch_ts", "priority").count().orderBy("batch_ts"))

### The other mode: `rescue`

`rescue` never fails. Unknown fields go into `_rescued_data` as JSON and the stream carries on -
you keep the data but the column doesn't appear in the table until you widen the schema yourself.

Trade-off in one line: `addNewColumns` gives you clean columns at the cost of one failed run,
`rescue` gives you uptime at the cost of data hiding in a text blob.

Separate folder, checkpoint and table for this, and only a few thousand rows - no point replaying
a million to make the point.

In [0]:
rescue_dir    = f"{landing_stream}/rescue_demo"
rescue_ckpt   = f"{chk}/rescue_demo"
rescue_target = f"{catalog}.{bronze}.rescue_demo_bronze"


def run_stream(source, ckpt_path, table, mode="rescue", max_files=2, trigger=None):
    """Same idea as run_autoloader, but every path is a parameter so the demos don't collide."""
    q = (spark.readStream.format("cloudFiles")
         .option("cloudFiles.format", "json")
         .option("cloudFiles.schemaLocation", f"{ckpt_path}/schema")
         .option("cloudFiles.schemaEvolutionMode", mode)
         .option("cloudFiles.maxFilesPerTrigger", max_files)
         .option("rescuedDataColumn", "_rescued_data")
         .load(source)
         .withColumn("source_file",  F.col("_metadata.file_path"))
         .withColumn("ingestion_ts", F.current_timestamp())
         .writeStream
         .option("checkpointLocation", ckpt_path)
         .option("mergeSchema", "true")
         .trigger(**(trigger or {"availableNow": True}))
         .toTable(table))
    q.awaitTermination()
    return q


small = src.limit(2000)
generate_batch(small, n_files=4, stamp="gen1", base=rescue_dir)
run_stream(rescue_dir, rescue_ckpt, rescue_target)
print("after gen1:", spark.table(rescue_target).count(), "rows")

In [0]:
# second generation with the extra column - no exception this time
generate_batch(small.withColumn("priority", F.lit("high")),
               n_files=4, stamp="gen2", base=rescue_dir)
run_stream(rescue_dir, rescue_ckpt, rescue_target)
print("after gen2:", spark.table(rescue_target).count(), "rows")

# priority is nowhere to be seen as a column - it's inside _rescued_data
print("columns:", spark.table(rescue_target).columns)
display(spark.table(rescue_target)
        .filter(F.col("_rescued_data").isNotNull())
        .select("batch_ts", "_rescued_data")
        .limit(5))

## 5. Triggers

Three ways to decide when a batch runs:

- **`availableNow`** - drain everything that's there, in as many batches as `maxFilesPerTrigger`
  dictates, then stop. What a scheduled job wants.
- **`once`** - one single batch, ignores `maxFilesPerTrigger`. Deprecated for exactly that reason:
  with 1000 files it tries to swallow the lot in one go and can blow up the cluster.
- **`processingTime`** - a batch every N seconds, forever. The cluster never goes idle, so you pay
  for it around the clock. Only worth it when latency actually matters.

Same 20 files through the first two, counting batches.

In [0]:
trigger_dir = f"{landing_stream}/trigger_demo"
generate_batch(src.limit(20_000), n_files=20, stamp="gen1", base=trigger_dir)

for name, trig in [("availablenow", {"availableNow": True}), ("once", {"once": True})]:
    table_t = f"{catalog}.{bronze}.trigger_{name}_bronze"
    q = run_stream(trigger_dir, f"{chk}/trigger_{name}", table_t, max_files=5, trigger=trig)
    print(f"{name:13} -> {len(q.recentProgress)} batches, {spark.table(table_t).count()} rows")

## 6. Reloading data safely

The checkpoint is the stream's memory. Three things live in there:

- `offsets/` + `commits/` - which batches were planned and which finished
- `sources/0/` - which files have already been seen (RocksDB, this is the dedup list)
- `schema/_schemas/` - the inferred schema, versioned

Delete it and the stream forgets everything, so it reads all the files again - but the table still
holds the old rows. That's how you end up with duplicates. Below: first the wrong way, then the
right one.

In [0]:
display(dbutils.fs.ls(rescue_ckpt))

In [0]:
# WRONG: checkpoint gone, table untouched -> everything gets read a second time
before = spark.table(rescue_target).count()

dbutils.fs.rm(rescue_ckpt, True)
run_stream(rescue_dir, rescue_ckpt, rescue_target)

after = spark.table(rescue_target).count()
print(f"{before} -> {after} rows ({after / before:.0f}x - duplicates)")

In [0]:
# RIGHT: wipe both sides together, then replay
dbutils.fs.rm(rescue_ckpt, True)
spark.sql(f"TRUNCATE TABLE {rescue_target}")
run_stream(rescue_dir, rescue_ckpt, rescue_target)

reloaded = spark.table(rescue_target).count()
print(f"reloaded: {reloaded} rows (expected {before})")
assert reloaded == before